# 02. Métricas, Baselines e MLflow - Telco Customer Churn

Este notebook consome o dataset final gerado pelo notebook 01 e implementa os baselines da Etapa 1.

Objetivos:

- validar o hash do dataset final versionado;
- definir métricas técnicas e métrica operacional/de negócio;
- treinar `DummyClassifier`, `LogisticRegression` e `RandomForestClassifier` com `Pipeline` sklearn;
- avaliar holdout e validação cruzada estratificada;
- escolher threshold por simulação econômica;
- registrar parâmetros, métricas, artefatos, modelos e versão do dataset no MLflow.

## 1. Configuração e contrato do dataset

A célula abaixo obtém o CSV e o manifesto da **última run** do experimento MLflow `telco-churn-dataset` (publicada no fim do notebook 01), valida o SHA256 do ficheiro contra o manifesto e prepara os objetos base. Opcional: defina `DATASET_MLFLOW_RUN_ID` para fixar uma run. Se o hash não bater, o notebook falha.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "target"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MLFLOW_TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').resolve().as_posix()}"
MLFLOW_EXPERIMENT_NAME = "telco-churn-etapa-1-baselines"
MLFLOW_DATASET_EXPERIMENT_NAME = "telco-churn-dataset"

logging.getLogger("mlflow.sklearn").setLevel(logging.ERROR)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_published_processed_dataset(
    *,
    tracking_uri: str,
    dataset_experiment: str,
) -> tuple[pd.DataFrame, dict, str]:
    """Última publicação do notebook 01 no MLflow, ou run fixada em DATASET_MLFLOW_RUN_ID."""
    mlflow.set_tracking_uri(tracking_uri)
    client = MlflowClient(tracking_uri=tracking_uri)
    run_override = (os.environ.get("DATASET_MLFLOW_RUN_ID") or "").strip()
    if run_override:
        run_id = run_override
    else:
        exp = client.get_experiment_by_name(dataset_experiment)
        if exp is None:
            raise RuntimeError(
                "Experimento de dataset ausente. Execute o notebook 01 até publicar "
                f"({dataset_experiment!r})."
            )
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            filter_string="tags.dataset_kind = 'telco_churn_model_ready'",
            order_by=["attributes.start_time DESC"],
            max_results=1,
        )
        if not runs:
            raise RuntimeError(
                "Nenhum dataset no MLflow. Complete o notebook 01 ou defina DATASET_MLFLOW_RUN_ID."
            )
        run_id = runs[0].info.run_id

    tmpdir = tempfile.mkdtemp(prefix="telco_dataset_")
    local_dataset_dir = Path(client.download_artifacts(run_id, "dataset", tmpdir))
    dataset_csv = local_dataset_dir / "telco_churn_model_ready.csv"
    manifest_json = local_dataset_dir / "telco_churn_model_ready_manifest.json"
    if not dataset_csv.is_file():
        raise FileNotFoundError(f"CSV ausente após download MLflow: {dataset_csv}")
    manifest_local = json.loads(manifest_json.read_text(encoding="utf-8"))
    actual_hash = sha256_file(dataset_csv)
    expected_hash = manifest_local["output_sha256"]
    if actual_hash != expected_hash:
        raise ValueError(f"Hash do dataset divergente: {actual_hash} != {expected_hash}")
    return pd.read_csv(dataset_csv), manifest_local, run_id


df, manifest, DATASET_SOURCE_RUN_ID = load_published_processed_dataset(
    tracking_uri=MLFLOW_TRACKING_URI,
    dataset_experiment=MLFLOW_DATASET_EXPERIMENT_NAME,
)
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

actual_hash = manifest["output_sha256"]

print(f"Dataset via MLflow (run_id): {DATASET_SOURCE_RUN_ID}")
print(f"SHA256 validado: {actual_hash}")
print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas")
display(df.head())
display(pd.Series(manifest).to_frame("manifest"))


Dataset: data/processed/telco_churn_model_ready.csv
SHA256 validado: e2b44e32aec0bc015c923deec5c00fddfdfaa0f6776efb0c9a22eaca7f4e3b74
Shape: 7,043 linhas x 42 colunas


,Gender,Age,Under30,SeniorCitizen,Married,Dependents,NumberofDependents,City,ZipCode,LatLong,...,PaymentMethod,MonthlyCharge,TotalCharges,TotalRefunds,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,Population,SatisfactionScore,target
0,Male,78,No,Yes,No,No,0,Los Angeles,90022,"34.02381, -118.156582",...,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,68701,3,1
1,Female,74,No,Yes,Yes,Yes,1,Los Angeles,90063,"34.044271, -118.185237",...,Credit Card,80.65,633.30,0.00,0,390.80,1024.10,55668,3,1
2,Male,71,No,Yes,No,Yes,3,Los Angeles,90065,"34.108833, -118.229715",...,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88,47534,2,1
3,Female,78,No,Yes,Yes,Yes,1,Inglewood,90303,"33.936291, -118.332639",...,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2995.07,27778,2,1
4,Female,80,No,Yes,Yes,Yes,1,Whittier,90602,"33.972119, -118.020188",...,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,3102.36,26265,2,1


,manifest
dataset_name,telco_churn_model_ready
dataset_version,e2b44e32aec0bc015c923deec5c00fddfdfaa0f6776efb...
created_at_utc,2026-05-03T02:03:38.610785+00:00
random_seed,42
source_type,raw_excel
source_files,"[Telco_customer_churn_demographics.xlsx, Telco..."
output_file,data/processed/telco_churn_model_ready.csv
output_sha256,e2b44e32aec0bc015c923deec5c00fddfdfaa0f6776efb...
target_column,target
rows,7043


## 2. Split estratificado e pré-processamento

O dataset final já remove leakage e IDs. Aqui só separamos `X`/`y`, criamos holdout estratificado e definimos um `ColumnTransformer` ajustado exclusivamente dentro do pipeline de treino.

In [2]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

print(f"Treino: {X_train.shape}; teste: {X_test.shape}")
print(f"Features numéricas/bool: {len(numeric_features)}")
print(f"Features categóricas: {len(categorical_features)}")
display(y.value_counts(normalize=True).rename("target_rate").to_frame())

Treino: (5634, 41); teste: (1409, 41)
Features numéricas/bool: 17
Features categóricas: 24


,target_rate
target,
0,0.73463
1,0.26537


## 3. Métricas e baselines

A métrica primária é `PR-AUC`, porque churn é a classe positiva minoritária e o uso operacional é priorizar clientes por risco. `ROC-AUC`, `F1`, `precision`, `recall`, matriz de confusão e `accuracy` entram como suporte diagnóstico.

In [3]:
def build_pipeline(model: object) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


def positive_scores(pipeline: Pipeline, X_eval: pd.DataFrame) -> np.ndarray:
    if hasattr(pipeline, "predict_proba"):
        return pipeline.predict_proba(X_eval)[:, 1]
    if hasattr(pipeline, "decision_function"):
        scores = pipeline.decision_function(X_eval)
        return 1 / (1 + np.exp(-scores))
    return pipeline.predict(X_eval).astype(float)


def evaluate_predictions(y_true: pd.Series, y_score: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_pred = (y_score >= threshold).astype(int)
    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics.update({"tn": tn, "fp": fp, "fn": fn, "tp": tp})
    return metrics


models = {
    "dummy_classifier": DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED),
    "logistic_regression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_SEED,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    ),
}

holdout_rows = []
fitted_pipelines = {}
for model_name, model in models.items():
    pipeline = build_pipeline(model)
    pipeline.fit(X_train, y_train)
    y_score = positive_scores(pipeline, X_test)
    row = {"model": model_name, **evaluate_predictions(y_test, y_score, threshold=0.5)}
    holdout_rows.append(row)
    fitted_pipelines[model_name] = pipeline

holdout_results = pd.DataFrame(holdout_rows).sort_values("pr_auc", ascending=False)
display(holdout_results)

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
1,logistic_regression,0.5,0.953868,0.905512,0.922460,0.913907,0.991315,0.980285,999,36,29,345
2,random_forest,0.5,0.924060,0.946488,0.756684,0.841010,0.967918,0.938433,1019,16,91,283
0,dummy_classifier,0.5,0.734564,0.000000,0.000000,0.000000,0.500000,0.265436,1035,0,374,0


## 4. Validação cruzada estratificada

A validação cruzada estratificada compara os baselines mantendo a proporção de churn em cada fold. Isso reduz o risco de decidir com base em uma única divisão treino/teste.

In [4]:
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_rows = []
for model_name, model in models.items():
    scores = cross_validate(
        build_pipeline(model),
        X,
        y,
        scoring=scoring,
        cv=cv,
        error_score="raise",
    )
    row = {"model": model_name}
    for metric_name in scoring:
        values = scores[f"test_{metric_name}"]
        row[f"cv_{metric_name}_mean"] = values.mean()
        row[f"cv_{metric_name}_std"] = values.std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("cv_pr_auc_mean", ascending=False)
display(cv_results)

,model,cv_pr_auc_mean,cv_pr_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_precision_mean,cv_precision_std,cv_recall_mean,cv_recall_std
1,logistic_regression,0.982609,0.002853,0.992344,0.001339,0.918155,0.008569,0.891294,0.019047,0.947029,0.007462
2,random_forest,0.947848,0.005518,0.974218,0.002178,0.849970,0.009517,0.943814,0.015765,0.773671,0.021082
0,dummy_classifier,0.265370,0.000239,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 5. Threshold e métrica de negócio

A regressão logística gera probabilidades. Esta célula busca o threshold com maior ganho esperado usando premissas explícitas: custo de contato/incentivo, meses de receita potencialmente retida e taxa de sucesso da ação de retenção. `MonthlyCharge` é usado como proxy observável de valor.

In [5]:
INTERVENTION_COST = 50.0
RETAINED_MONTHS = 6
RETENTION_SUCCESS_RATE = 0.25
monthly_revenue_proxy = float(X_test["MonthlyCharge"].mean()) if "MonthlyCharge" in X_test.columns else 70.0
avoidable_loss_proxy = monthly_revenue_proxy * RETAINED_MONTHS * RETENTION_SUCCESS_RATE
TP_GAIN = avoidable_loss_proxy - INTERVENTION_COST
FP_GAIN = -INTERVENTION_COST
FN_GAIN = -avoidable_loss_proxy
TN_GAIN = 0.0

business_assumptions = {
    "intervention_cost": INTERVENTION_COST,
    "retained_months": RETAINED_MONTHS,
    "retention_success_rate": RETENTION_SUCCESS_RATE,
    "monthly_revenue_proxy": monthly_revenue_proxy,
    "tp_gain": TP_GAIN,
    "fp_gain": FP_GAIN,
    "fn_gain": FN_GAIN,
    "tn_gain": TN_GAIN,
}

def expected_business_value(y_true: pd.Series, y_score: np.ndarray, threshold: float) -> dict[str, float]:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total_value = tp * TP_GAIN + fp * FP_GAIN + fn * FN_GAIN + tn * TN_GAIN
    return {
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "expected_value": total_value,
        "expected_value_per_customer": total_value / len(y_true),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

best_model_name = holdout_results.iloc[0]["model"]
best_pipeline = fitted_pipelines[best_model_name]
best_scores = positive_scores(best_pipeline, X_test)
threshold_grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
threshold_results = pd.DataFrame(
    [expected_business_value(y_test, best_scores, threshold) for threshold in threshold_grid]
).sort_values("expected_value", ascending=False)
best_threshold_row = threshold_results.iloc[0]
best_threshold = float(best_threshold_row["threshold"])

print(f"Melhor modelo por PR-AUC holdout: {best_model_name}")
print(f"Melhor threshold por valor esperado: {best_threshold:.2f}")
display(pd.Series(business_assumptions, name="value").to_frame())
display(threshold_results.head(10))

final_metrics = evaluate_predictions(y_test, best_scores, threshold=best_threshold)
final_confusion = pd.DataFrame(
    confusion_matrix(y_test, (best_scores >= best_threshold).astype(int), labels=[0, 1]),
    index=["real_nao_churn", "real_churn"],
    columns=["pred_nao_churn", "pred_churn"],
)
display(pd.Series(final_metrics, name="final_metric").to_frame())
display(final_confusion)

Melhor modelo por PR-AUC holdout: logistic_regression
Melhor threshold por valor esperado: 0.38


,value
intervention_cost,50.000000
retained_months,6.000000
retention_success_rate,0.250000
monthly_revenue_proxy,64.244535
tp_gain,46.366803
fp_gain,-50.000000
fn_gain,-96.366803
tn_gain,0.000000


,threshold,tn,fp,fn,tp,expected_value,expected_value_per_customer,precision,recall,f1
33,0.38,983,52,20,354,11886.512101,8.436133,0.871921,0.946524,0.907692
35,0.40,985,50,21,353,11843.778495,8.405804,0.875931,0.943850,0.908623
26,0.31,967,68,15,359,11800.180128,8.374862,0.840749,0.959893,0.896380
34,0.39,984,51,21,353,11793.778495,8.370318,0.873762,0.943850,0.907455
25,0.30,966,69,15,359,11750.180128,8.339376,0.838785,0.959893,0.895262
32,0.37,980,55,20,354,11736.512101,8.329675,0.865526,0.946524,0.904215
36,0.41,985,50,22,352,11701.044890,8.304503,0.875622,0.941176,0.907216
31,0.36,979,56,20,354,11686.512101,8.294189,0.863415,0.946524,0.903061
30,0.35,976,59,19,355,11679.245706,8.289032,0.857488,0.949198,0.901015
28,0.33,973,62,18,356,11671.979312,8.283875,0.851675,0.951872,0.898990


,final_metric
threshold,0.380000
accuracy,0.948900
precision,0.871921
recall,0.946524
f1,0.907692
roc_auc,0.991315
pr_auc,0.980285
tn,983.000000
fp,52.000000
fn,20.000000


,pred_nao_churn,pred_churn
real_nao_churn,983,52
real_churn,20,354


## 6. Registro no MLflow

In [6]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


def make_mlflow_input_example(sample: pd.DataFrame) -> pd.DataFrame:
    example = sample.head(5).copy()
    numeric_columns = example.select_dtypes(include=["number", "bool"]).columns
    example[numeric_columns] = example[numeric_columns].astype("float64")
    return example


input_example = make_mlflow_input_example(X_train)
cv_by_model = cv_results.set_index("model").to_dict(orient="index")
holdout_by_model = holdout_results.set_index("model").to_dict(orient="index")

for model_name, pipeline in fitted_pipelines.items():
    with mlflow.start_run(run_name=f"etapa1-{model_name}"):
        mlflow.set_tags(
            {
                "dataset.name": manifest["dataset_name"],
                "dataset.version": manifest["dataset_version"],
                "dataset.hash": manifest["output_sha256"],
                "dataset.source_type": manifest["source_type"],
                "dataset.mlflow_run_id": DATASET_SOURCE_RUN_ID,
                "stage": "etapa_1_baseline",
                "baseline.key": model_name,
            }
        )
        mlflow.log_params(
            {
                "model_name": model_name,
                "random_seed": RANDOM_SEED,
                "test_size": TEST_SIZE,
                "target_column": TARGET_COLUMN,
                "n_rows": len(df),
                "n_features": X.shape[1],
                "primary_metric": "pr_auc",
            }
        )
        model_params = pipeline.named_steps["model"].get_params()
        mlflow.log_params(
            {
                f"model__{key}": value
                for key, value in model_params.items()
                if isinstance(value, (str, int, float, bool, type(None)))
            }
        )
        mlflow.log_metrics({f"holdout_{k}": v for k, v in holdout_by_model[model_name].items()})
        mlflow.log_metrics({k: v for k, v in cv_by_model[model_name].items()})

        if model_name == best_model_name:
            mlflow.log_params({f"business__{k}": v for k, v in business_assumptions.items()})
            mlflow.log_metrics({f"final_{k}": v for k, v in final_metrics.items()})
            mlflow.log_metric("business_expected_value", float(best_threshold_row["expected_value"]))
            mlflow.log_metric(
                "business_expected_value_per_customer",
                float(best_threshold_row["expected_value_per_customer"]),
            )

        with tempfile.TemporaryDirectory() as temp_dir:
            temp_path = Path(temp_dir)
            manifest_artifact = temp_path / "dataset_manifest.json"
            manifest_artifact.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
            holdout_results.to_csv(temp_path / "holdout_results.csv", index=False)
            cv_results.to_csv(temp_path / "cv_results.csv", index=False)
            threshold_results.to_csv(temp_path / "threshold_results.csv", index=False)
            final_confusion.to_csv(temp_path / "final_confusion_matrix.csv")

            mlflow.log_artifact(str(manifest_artifact), artifact_path="dataset")
            mlflow.log_artifact(str(temp_path / "holdout_results.csv"), artifact_path="metrics")
            mlflow.log_artifact(str(temp_path / "cv_results.csv"), artifact_path="metrics")
            mlflow.log_artifact(str(temp_path / "threshold_results.csv"), artifact_path="business")
            mlflow.log_artifact(str(temp_path / "final_confusion_matrix.csv"), artifact_path="metrics")

        model_signature = infer_signature(
            input_example,
            positive_scores(pipeline, input_example).astype("float64"),
        )
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            input_example=input_example,
            signature=model_signature,
        )

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experimento: {MLFLOW_EXPERIMENT_NAME}")

MLflow tracking URI: sqlite:////Users/crisley/Documents/postech-ml-challenge-fase-1/mlflow.db
Experimento: telco-churn-etapa-1-baselines


## Conclusão

In [7]:
decision_table = (
    holdout_results.merge(cv_results, on="model", how="left")
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)
display(decision_table)

summary = {
    "best_model_by_pr_auc": best_model_name,
    "primary_metric": "PR-AUC / average_precision",
    "best_threshold_by_business_value": best_threshold,
    "expected_value": float(best_threshold_row["expected_value"]),
    "expected_value_per_customer": float(best_threshold_row["expected_value_per_customer"]),
    "dataset_version": manifest["dataset_version"],
}
display(pd.Series(summary, name="decision").to_frame())


,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,...,cv_pr_auc_mean,cv_pr_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_precision_mean,cv_precision_std,cv_recall_mean,cv_recall_std
0,logistic_regression,0.5,0.953868,0.905512,0.922460,0.913907,0.991315,0.980285,999,36,...,0.982609,0.002853,0.992344,0.001339,0.918155,0.008569,0.891294,0.019047,0.947029,0.007462
1,random_forest,0.5,0.924060,0.946488,0.756684,0.841010,0.967918,0.938433,1019,16,...,0.947848,0.005518,0.974218,0.002178,0.849970,0.009517,0.943814,0.015765,0.773671,0.021082
2,dummy_classifier,0.5,0.734564,0.000000,0.000000,0.000000,0.500000,0.265436,1035,0,...,0.265370,0.000239,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,decision
best_model_by_pr_auc,logistic_regression
primary_metric,PR-AUC / average_precision
best_threshold_by_business_value,0.38
expected_value,11886.512101
expected_value_per_customer,8.436133
dataset_version,e2b44e32aec0bc015c923deec5c00fddfdfaa0f6776efb...
